# Ch 5: Building a Vision-Language Model from Scratch (Qwen-VL Style)

**Papers**: Qwen2.5-VL (Bai 2025) · InternVL3 (Chen 2025) · LLaVA-1.5 (Liu 2023)

**What you'll build**: A *complete* VLM pipeline — vision encoder, MLP projector, multimodal
rotary position embeddings (mRoPE), visual token insertion, attention masking, loss masking,
and a Stage 1 training loop — all implemented from scratch.

| Component | Implementation | Rationale |
|---|---|---|
| TinyViT (SigLIP-style) | **from scratch** | understand patch encoding + SwiGLU |
| MLP Projector | **from scratch** | the sole Stage-1 trainable bridge |
| mRoPE | **from scratch** | key novelty of Qwen2.5-VL |
| Token insertion + masking | **from scratch** | core multimodal plumbing |
| Tokenizer | HuggingFace (`bert-base-uncased` vocab) | off-topic boilerplate |

> **Runtime**: CPU is fine for all demos; a T4 GPU accelerates Stage 1 training.


In [ ]:
# Install lightweight dependencies — no large model downloads required
!pip install -q transformers torch pillow numpy matplotlib


In [ ]:
import json, math, random, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


# 5.1 Architecture Overview & Component Assembly

## Intuition

A VLM answers the question: *"how do we give a text-generating LLM the ability to see?"*

The answer has three parts:

1. **Vision encoder** – Chop an image into patches; transform them into dense vectors.
   SigLIP (used by Qwen2.5-VL) is a ViT trained with a sigmoid contrastive loss
   rather than softmax, producing richer per-patch representations.

2. **MLP projector** – Map patch vectors from `visual_dim → llm_dim`.
   The LLM and ViT were trained independently; their embedding spaces are misaligned.
   The projector is the *learned translator*. During Stage 1, it is the **only**
   component trained; everything else stays frozen.

3. **Language model** – A standard causal transformer that sees a flattened token
   sequence of the form:
   ```
   [<|vision_start|>] [v₁ … vₙ] [<|vision_end|>] [text tokens …]
   ```
   The multimodal rotary position embedding (mRoPE) gives each token a 3-D position
   `(time, height, width)` so the LLM can reason about spatial layout.

## Sample I/O (tiny demo model)
| Stage | Tensor shape | Meaning |
|---|---|---|
| Raw image | `(1, 3, 64, 64)` | 64×64 RGB image |
| After TinyViT | `(1, 16, 256)` | 16 patch tokens, 256-d each |
| After Projector | `(1, 16, 192)` | aligned to LLM dim |
| Full sequence | `(1, 20, 192)` | `<vis_start>` + 16 + `<vis_end>` + 2 text |
| LLM output logits | `(1, 20, 30522)` | per-token vocabulary logits |


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Central configuration dataclass — single source of truth for all
# hyper-parameters. Every sub-module reads from this object.
# ──────────────────────────────────────────────────────────────────
@dataclass
class VLMConfig:
    # ── Vision encoder ──────────────────────────────────────────
    image_size: int = 64          # H = W (square for simplicity)
    patch_size: int = 16          # each patch covers 16×16 pixels
    visual_dim: int = 256         # ViT hidden / output dimension
    visual_num_heads: int = 4
    visual_num_layers: int = 3

    # ── MLP projector ───────────────────────────────────────────
    projector_hidden_dim: int = 512

    # ── Language model ──────────────────────────────────────────
    # llm_dim must satisfy: (llm_dim // llm_num_heads) % 6 == 0
    # so that head_dim splits evenly into 3 mRoPE components.
    # 192 / 4 = 48; 48 / 6 = 8 ✓
    llm_dim: int = 192
    llm_num_heads: int = 4        # head_dim = 48
    llm_num_layers: int = 4
    llm_ffn_dim: int = 384
    vocab_size: int = 30522       # BERT vocab (bert-base-uncased)
    max_seq_len: int = 512

    # ── mRoPE ───────────────────────────────────────────────────
    rope_base: float = 10000.0

    # ── Special token IDs ────────────────────────────────────────
    # BERT [unusedX] tokens — safe to repurpose, never in real text.
    vision_start_id: int = 1
    vision_end_id: int = 2
    image_pad_id: int = 3
    pad_token_id: int = 0

    @property
    def num_patches(self) -> int:
        return (self.image_size // self.patch_size) ** 2      # 16

    @property
    def patch_grid_hw(self) -> Tuple[int, int]:
        g = self.image_size // self.patch_size
        return (g, g)                                          # (4, 4)

    @property
    def head_dim(self) -> int:
        return self.llm_dim // self.llm_num_heads              # 48


cfg = VLMConfig()
print(f"Patch grid : {cfg.patch_grid_hw}  →  {cfg.num_patches} tokens per image")
print(f"head_dim   : {cfg.head_dim}  (must be divisible by 6: {cfg.head_dim % 6 == 0})")


## 5.1.1 Vision Encoder — TinyViT (SigLIP-style)

### Key design decisions vs. vanilla ViT

| Choice | Vanilla ViT | SigLIP / Qwen2.5-VL | Reason |
|---|---|---|---|
| Activation | GELU | **SwiGLU** | higher capacity per parameter |
| Pooling | [CLS] token | **all patch tokens** | preserves spatial layout for grounding |
| Contrastive loss | softmax | sigmoid | allows multiple matches per image |
| Pos. embed | learned absolute | Qwen uses **mRoPE** inside LLM | flexibility for variable resolution |

**SwiGLU**: `FFN(x) = down(SiLU(gate(x)) ⊙ up(x))`
The *gate* path learns *what* information to pass through; the *up* path carries the actual content.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# SwiGLU feed-forward block used inside the ViT.
# Replaces standard MLP: SiLU gate controls information flow.
# ──────────────────────────────────────────────────────────────────
class SwiGLU(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.up_proj   = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, embed_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, num_tokens, embed_dim) → (batch_num, num_tokens, hidden_dim)
        gate = F.silu(self.gate_proj(x))
        # (batch_num, num_tokens, embed_dim) → (batch_num, num_tokens, hidden_dim)
        up   = self.up_proj(x)
        # Element-wise gate; (batch_num, num_tokens, hidden_dim) → (batch_num, num_tokens, embed_dim)
        return self.down_proj(gate * up)


# ──────────────────────────────────────────────────────────────────
# Standard multi-head self-attention for the ViT (bidirectional —
# no causal mask; every patch can attend to every other patch).
# ──────────────────────────────────────────────────────────────────
class ViTAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv  = nn.Linear(embed_dim, 3 * embed_dim, bias=True)
        self.proj = nn.Linear(embed_dim, embed_dim, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, num_tokens, embed_dim = x.shape

        # Fused QKV projection, then reshape for multi-head split
        # (batch_num, num_tokens, embed_dim) → (batch_num, num_tokens, 3*embed_dim)
        qkv = self.qkv(x)
        # (batch_num, num_tokens, 3, num_heads, head_dim) → (3, batch_num, num_heads, num_tokens, head_dim)
        qkv = qkv.reshape(batch_num, num_tokens, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        # Scaled dot-product attention — fully bidirectional (no mask)
        # (batch_num, num_heads, num_tokens, num_tokens)
        attn = F.softmax((q @ k.transpose(-2, -1)) * self.scale, dim=-1)

        # (batch_num, num_heads, num_tokens, head_dim) → (batch_num, num_tokens, embed_dim)
        x = (attn @ v).transpose(1, 2).reshape(batch_num, num_tokens, embed_dim)
        # (batch_num, num_tokens, embed_dim) → (batch_num, num_tokens, embed_dim)
        return self.proj(x)


# ──────────────────────────────────────────────────────────────────
# Single ViT transformer block: pre-norm design for training stability.
# Pre-norm (norm before sub-layer) vs post-norm: avoids gradient explosion.
# ──────────────────────────────────────────────────────────────────
class ViTBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, ffn_hidden: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = ViTAttention(embed_dim, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = SwiGLU(embed_dim, ffn_hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm attention with residual
        # (batch_num, num_tokens, embed_dim) → (batch_num, num_tokens, embed_dim)
        x = x + self.attn(self.norm1(x))
        # Pre-norm SwiGLU FFN with residual
        # (batch_num, num_tokens, embed_dim) → (batch_num, num_tokens, embed_dim)
        x = x + self.ffn(self.norm2(x))
        return x


# ──────────────────────────────────────────────────────────────────
# TinyViT: SigLIP-style ViT, returns ALL patch tokens (no CLS token).
# Production replacement: google/siglip-so400m-patch14-384 — same API.
# ──────────────────────────────────────────────────────────────────
class TinyViT(nn.Module):
    """
    Minimal SigLIP-style Vision Transformer.

    Differences from vanilla ViT:
    - NO [CLS] token  → returns spatial patch features for grounding
    - SwiGLU FFN      → higher capacity per parameter
    - Conv2d patch embed → equivalent to linear but GPU-friendly

    Swap-in for production:
        from transformers import SiglipVisionModel
        vit = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224")
        visual_features = vit(pixel_values=images).last_hidden_state
    """

    def __init__(self, cfg: VLMConfig):
        super().__init__()
        self.patch_size  = cfg.patch_size
        self.num_patches = cfg.num_patches

        # Conv2d patch embedding: (B,3,H,W) → (B,visual_dim,H/p,W/p)
        self.patch_embed = nn.Conv2d(
            3, cfg.visual_dim, kernel_size=cfg.patch_size, stride=cfg.patch_size
        )

        # Absolute positional embedding (one vector per patch position)
        self.pos_embed = nn.Parameter(
            torch.randn(1, cfg.num_patches, cfg.visual_dim) * 0.02
        )

        # SwiGLU hidden dim: using 4x width, halved because SwiGLU has 2 projections
        ffn_hidden = cfg.visual_dim * 2
        self.blocks = nn.ModuleList([
            ViTBlock(cfg.visual_dim, cfg.visual_num_heads, ffn_hidden)
            for _ in range(cfg.visual_num_layers)
        ])
        self.norm = nn.LayerNorm(cfg.visual_dim)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pixel_values: (batch_num, 3, image_size, image_size)
        Returns:
            visual_features: (batch_num, num_patches, visual_dim)
        """
        # Embed patches: (batch_num, 3, H, W) → (batch_num, visual_dim, H/p, W/p)
        x = self.patch_embed(pixel_values)
        # Flatten spatial dims and transpose: → (batch_num, num_patches, visual_dim)
        x = x.flatten(2).transpose(1, 2)

        # Add spatial positional bias (broadcast over batch)
        # (batch_num, num_patches, visual_dim) + (1, num_patches, visual_dim)
        x = x + self.pos_embed

        # Apply transformer encoder blocks
        for block in self.blocks:
            # (batch_num, num_patches, visual_dim) → (batch_num, num_patches, visual_dim)
            x = block(x)

        # Final layer norm
        # (batch_num, num_patches, visual_dim) → (batch_num, num_patches, visual_dim)
        return self.norm(x)


# ── Sanity check ─────────────────────────────────────────────────
vit     = TinyViT(cfg).to(DEVICE)
dummy   = torch.randn(2, 3, cfg.image_size, cfg.image_size).to(DEVICE)
vis_out = vit(dummy)
print(f"TinyViT  {tuple(dummy.shape)} → {tuple(vis_out.shape)}")
print(f"Params   {sum(p.numel() for p in vit.parameters()):,}")


## 5.1.2 MLP Projector — Trainable Bridge (Stage 1 only)

### Why not a single linear layer?

A single linear map `W: R^{visual_dim} → R^{llm_dim}` can only learn
*rotations and scalings* of the visual feature space.  
The ViT and LLM have completely different training objectives; their spaces are
non-linearly misaligned. The 2-layer MLP with GELU introduces the nonlinearity
needed to learn a more expressive alignment.

**Layer norm** after the second projection stabilises early training — gradient
norms through the LLM embedding lookup can be large initially.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# 2-layer MLP projector: the ONLY trainable component in Stage 1.
# Maps visual patch features into the LLM's embedding dimension.
# ──────────────────────────────────────────────────────────────────
class MLPProjector(nn.Module):
    """
    visual_dim → hidden_dim → llm_dim  (with GELU + LayerNorm)

    Stage 1 role: translate frozen ViT features into the LLM's token space.
    Stage 2+ role: continues to train alongside the LLM.
    """

    def __init__(self, visual_dim: int, llm_dim: int, hidden_dim: int):
        super().__init__()
        self.fc1  = nn.Linear(visual_dim, hidden_dim)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(hidden_dim, llm_dim)
        # Post-projection norm stabilises LLM input magnitudes
        self.norm = nn.LayerNorm(llm_dim)

    def forward(self, visual_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            visual_features: (batch_num, num_patches, visual_dim)
        Returns:
            projected: (batch_num, num_patches, llm_dim)
        """
        # (batch_num, num_patches, visual_dim) → (batch_num, num_patches, hidden_dim)
        x = self.fc1(visual_features)
        # (batch_num, num_patches, hidden_dim) → (batch_num, num_patches, hidden_dim)
        x = self.act(x)
        # (batch_num, num_patches, hidden_dim) → (batch_num, num_patches, llm_dim)
        x = self.fc2(x)
        # (batch_num, num_patches, llm_dim) → (batch_num, num_patches, llm_dim)
        return self.norm(x)


projector   = MLPProjector(cfg.visual_dim, cfg.llm_dim, cfg.projector_hidden_dim).to(DEVICE)
proj_out    = projector(vis_out)
print(f"Projector {tuple(vis_out.shape)} → {tuple(proj_out.shape)}")
print(f"Params    {sum(p.numel() for p in projector.parameters()):,}")


# 5.2 Data Pipeline: Dynamic Resolution & Tokenization

## Intuition

Real VLMs handle images of *arbitrary* resolution. A 224×224 thumbnail and a
1344×896 document scan should both be processable. The number of patch tokens
scales with image area: `N = (H/p) × (W/p)`.

For Stage 1 we use a **synthetic dataset**: solid-colored 64×64 images paired
with color-name captions. This is small enough to overfit quickly,
letting us *verify* the training loop without downloading any data.

## Synthetic Dataset

| Color | RGB | Caption |
|---|---|---|
| red | (220, 50, 50) | `"this image shows a red background"` |
| blue | (50, 100, 220) | `"this image shows a blue background"` |
| … | … | … |

**Why synthetic?** We need to verify that the model can learn *something* after
Stage 1. If it can learn to caption solid colors, the architecture, data pipeline,
attention masking, and loss masking all work correctly.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Color palette used for synthetic image generation.
# Simple enough to overfit; varied enough to test generalisation.
# ──────────────────────────────────────────────────────────────────
COLORS: Dict[str, Tuple[int, int, int]] = {
    "red":    (220,  50,  50),
    "blue":   ( 50, 100, 220),
    "green":  ( 50, 180,  50),
    "yellow": (220, 200,  50),
    "purple": (150,  50, 200),
    "orange": (220, 130,  50),
    "pink":   (220, 100, 150),
    "cyan":   ( 50, 200, 200),
    "white":  (240, 240, 240),
    "gray":   (128, 128, 128),
}


class SyntheticColorDataset(Dataset):
    """
    Synthetic (image, caption) pairs — no downloads required.

    Image : 64×64 solid color + Gaussian noise (prevents trivial pixel shortcuts)
    Caption: "this image shows a <color> background"
    """

    def __init__(self, num_samples: int = 300, image_size: int = 64):
        self.image_size  = image_size
        self.color_names = list(COLORS.keys())
        rng = random.Random(42)
        self.samples = [rng.choice(self.color_names) for _ in range(num_samples)]

    def __len__(self) -> int:
        return len(self.samples)

    def _make_image(self, color_name: str) -> torch.Tensor:
        rgb = COLORS[color_name]
        # Fill canvas with target color + small Gaussian noise
        # (image_size, image_size, 3) normalised to [0, 1]
        canvas = np.full((self.image_size, self.image_size, 3), rgb, dtype=np.float32)
        canvas += np.random.randn(*canvas.shape).astype(np.float32) * 8.0
        canvas  = np.clip(canvas, 0.0, 255.0) / 255.0
        # (image_size, image_size, 3) → (3, image_size, image_size)
        return torch.from_numpy(canvas.transpose(2, 0, 1))

    def __getitem__(self, idx: int) -> Dict[str, object]:
        color = self.samples[idx]
        return {
            "image":   self._make_image(color),        # (3, H, W)
            "caption": f"this image shows a {color} background",
            "color":   color,
        }


# Preview the dataset
dataset = SyntheticColorDataset(num_samples=300, image_size=cfg.image_size)
sample  = dataset[0]
print(f"Image  shape : {sample['image'].shape}")
print(f"Caption      : {sample['caption']!r}")
print(f"Dataset size : {len(dataset)}")

# Visualise a few samples
fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
for ax, idx in zip(axes, [0, 30, 60, 90, 120]):
    s = dataset[idx]
    ax.imshow(s["image"].permute(1, 2, 0).numpy())
    ax.set_title(s["color"], fontsize=9)
    ax.axis("off")
plt.suptitle("Synthetic Training Images", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Load BERT tokenizer for its 30 522-token vocabulary.
# Only the vocabulary file (~500 KB) is downloaded — no model weights.
# ──────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Verify the special-token IDs we reserved in VLMConfig
print("Special token IDs:")
for name, tok_id in [
    ("<|vision_start|>", cfg.vision_start_id),
    ("<|vision_end|>",   cfg.vision_end_id),
    ("<|image_pad|>",    cfg.image_pad_id),
]:
    decoded = tokenizer.decode([tok_id])
    print(f"  {name:20s} id={tok_id:5d}  decoded={decoded!r}")

# Quick tokenisation sanity check
test_ids = tokenizer.encode("this image shows a red background", add_special_tokens=False)
print(f"\nSample caption tokens: {test_ids}")
print(f"Decoded back         : {tokenizer.decode(test_ids)!r}")


# 5.3 Visual Token Insertion

## Intuition

The LLM operates on a flat 1-D token sequence. Visual tokens must be *injected*
into that sequence in a way the LLM can distinguish from regular text.
Qwen2.5-VL uses three special tokens:

```
<|vision_start|>  v₁  v₂  …  vₙ  <|vision_end|>  text tokens …
      ↑                                  ↑
  boundary marker             boundary marker
```

After token-ID construction, the embedding layer replaces:
- `vision_start_id`, `vision_end_id`, `image_pad_id` → their token embeddings
- The actual visual patch embeddings overwrite the `image_pad_id` positions at
  embedding time (not at token-ID time).

### Why separate `<|image_pad|>` tokens?

The `image_pad_id` serves as a **placeholder** in the token-ID tensor. During the
forward pass, the LLM's `nn.Embedding` lookup produces dummy embeddings at these
positions, which are then *overwritten* by the projected visual features.
This keeps the token-ID tensor purely integer-typed and avoids mixing floats into
the discrete sequence construction step.

## Sample I/O

```
Input : num_patches=16, caption_ids=[4, 5, 6, 7, 8]
Output: [1, 3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3, 2, 4, 5, 6, 7, 8]
        ↑  ←────── 16 image_pad placeholders ──────→  ↑  ← text →
     vis_start                                      vis_end
```


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Build the mixed token-ID sequence:
# [<vis_start>] + [image_pad × N_patches] + [<vis_end>] + [text_ids]
#
# The image_pad slots will be overwritten with real visual embeddings
# inside the VLM forward pass.
# ──────────────────────────────────────────────────────────────────
def build_token_ids_with_visual(
    caption_ids: List[int],
    num_patches: int,
    cfg: VLMConfig,
) -> Tuple[torch.Tensor, int, int]:
    """
    Construct the interleaved token-ID sequence for a single example.

    Returns:
        token_ids  : (seq_len,) int64 token IDs
        vis_start  : index of <|vision_start|> in token_ids
        vis_end    : index of <|vision_end|>   in token_ids
    """
    vis_start_idx = 0
    # Placeholder patch positions immediately after the start marker
    patch_ids     = [cfg.image_pad_id] * num_patches
    vis_end_idx   = 1 + num_patches

    id_list = (
        [cfg.vision_start_id]
        + patch_ids
        + [cfg.vision_end_id]
        + caption_ids
    )
    # (seq_len,)
    token_ids = torch.tensor(id_list, dtype=torch.long)
    return token_ids, vis_start_idx, vis_end_idx


# ──────────────────────────────────────────────────────────────────
# Collate a batch into padded tensors, tracking visual boundaries.
# ──────────────────────────────────────────────────────────────────
def collate_vlm_batch(
    examples: List[Dict],
    tokenizer,
    cfg: VLMConfig,
) -> Dict[str, torch.Tensor]:
    """
    Args:
        examples : list of {'image': Tensor, 'caption': str} dicts
    Returns dict with keys:
        pixel_values : (batch_num, 3, H, W)
        input_ids    : (batch_num, seq_len)  – padded
        labels       : (batch_num, seq_len)  – -100 except caption tokens
        vis_ranges   : (batch_num, 2)         – (vis_start, vis_end) indices
    """
    batch_num = len(examples)

    # Stack images into a single batch tensor
    # (batch_num, 3, image_size, image_size)
    pixel_values = torch.stack([ex["image"] for ex in examples])

    # Tokenise captions (no special tokens — we insert our own)
    all_token_ids, all_vis_ranges, all_labels = [], [], []

    for ex in examples:
        cap_ids = tokenizer.encode(ex["caption"], add_special_tokens=False)
        token_ids, v_start, v_end = build_token_ids_with_visual(
            cap_ids, cfg.num_patches, cfg
        )

        # Label mask: -100 suppresses loss on everything except the caption
        labels = torch.full_like(token_ids, -100)
        caption_start = v_end + 1
        labels[caption_start:] = token_ids[caption_start:]

        all_token_ids.append(token_ids)
        all_vis_ranges.append((v_start, v_end))
        all_labels.append(labels)

    # Pad sequences to the longest in this batch
    max_len = max(t.size(0) for t in all_token_ids)

    def pad(tensor_list: List[torch.Tensor], pad_val: int) -> torch.Tensor:
        padded = torch.full((batch_num, max_len), pad_val, dtype=torch.long)
        for i, t in enumerate(tensor_list):
            padded[i, : t.size(0)] = t
        return padded

    return {
        "pixel_values": pixel_values,                               # (batch_num, 3, H, W)
        "input_ids":    pad(all_token_ids, cfg.pad_token_id),       # (batch_num, seq_len)
        "labels":       pad(all_labels, -100),                       # (batch_num, seq_len)
        "vis_ranges":   torch.tensor(all_vis_ranges, dtype=torch.long),  # (batch_num, 2)
    }


# ── Demo ──────────────────────────────────────────────────────────
cap_ids = tokenizer.encode("this image shows a red background", add_special_tokens=False)
token_ids, v_start, v_end = build_token_ids_with_visual(cap_ids, cfg.num_patches, cfg)

print(f"Sequence length : {token_ids.size(0)}")
print(f"  [0]           : {token_ids[0].item()} = <|vision_start|>")
print(f"  [1 .. {v_end-1}]   : {token_ids[1:v_end].tolist()} = image_pad × {cfg.num_patches}")
print(f"  [{v_end}]          : {token_ids[v_end].item()} = <|vision_end|>")
print(f"  [{v_end+1} ..]     : {token_ids[v_end+1:].tolist()} = caption tokens")


# 5.4 Multimodal Rotary Position Embeddings (mRoPE)

## Intuition

Standard 1-D RoPE encodes a single integer position for each token.
For images, position has *two* spatial dimensions (row, col). For video it has
three: (time, row, col). Qwen2.5-VL's **mRoPE** extends RoPE to 3-D:

```
position_ids shape: (3, seq_len)
  dim 0 → time   positions
  dim 1 → height positions
  dim 2 → width  positions
```

The attention head dimension `head_dim` is split into **3 equal sections**; each
section receives RoPE computed from one of the three coordinates.

### Position assignment rules

| Token type | time | height | width |
|---|---|---|---|
| `<\|vision_start\|>` | 0 | 0 | 0 |
| visual patch at row r, col c | 0 | r | c |
| `<\|vision_end\|>` | 0 | H_grid | W_grid |
| text token at position t | t | t | t |

Text tokens use the **same value** for all three dimensions — this degenerates
to standard 1-D RoPE for text, maintaining compatibility with a pretrained LLM.

## Sample I/O

```
Grid: 4×4 image, text length 5
position_ids: shape (3, 22)   [1 + 16 patches + 1 + 5 text − 1 for 0-index]
  time   : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 8, 9]
  height : [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8, 9]
  width  : [0, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 5, 6, 7, 8, 9]
```


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Compute inverse frequency bands for one mRoPE component.
# These are shared across time, height, and width (same base freq).
# ──────────────────────────────────────────────────────────────────
def get_rope_inv_freq(
    dim: int,
    base: float = 10000.0,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """
    Standard RoPE inverse frequencies for a single dimension.

    The formula θ_i = base^{-2i/dim} gives exponentially spaced frequencies,
    enabling the model to encode both fine-grained and coarse position differences.

    Args:
        dim : number of dimensions for this component (head_dim // 3)
    Returns:
        inv_freq: (dim // 2,)  — one frequency per rotation pair
    """
    # (dim // 2,)  — indices 0, 2, 4, …, dim-2
    exponents = torch.arange(0, dim, 2, dtype=torch.float32, device=device)
    # (dim // 2,)
    inv_freq  = 1.0 / (base ** (exponents / dim))
    return inv_freq


# ──────────────────────────────────────────────────────────────────
# Compute 3D position IDs for the full interleaved sequence.
# Sequence layout: [vis_start | patches | vis_end | text tokens]
# ──────────────────────────────────────────────────────────────────
def compute_mrope_position_ids(
    patch_grid_hw: Tuple[int, int],
    text_length: int,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """
    Build the (3, seq_len) position-ID matrix.

    Args:
        patch_grid_hw : (H_grid, W_grid) — e.g. (4, 4) for 16 patches
        text_length   : number of text tokens following <|vision_end|>
    Returns:
        position_ids  : (3, seq_len)  — rows are [time, height, width]
    """
    H, W = patch_grid_hw
    num_patches = H * W

    # ── <|vision_start|> ──────────────────────────────────────────
    # Anchor at origin; the LLM learns that this marks image start.
    # (3, 1)
    vis_start_pos = torch.zeros(3, 1, dtype=torch.long, device=device)

    # ── Visual patch tokens: 2-D spatial grid ─────────────────────
    # time=0 (static image, not video)
    # height: repeats each row W times   [0,0,0,0, 1,1,1,1, 2,2,2,2, 3,3,3,3]
    # width : cycles 0..W-1 for each row [0,1,2,3, 0,1,2,3, 0,1,2,3, 0,1,2,3]
    time_ids   = torch.zeros(num_patches, dtype=torch.long, device=device)
    height_ids = torch.arange(H, device=device).repeat_interleave(W)
    width_ids  = torch.arange(W, device=device).repeat(H)
    # (3, num_patches)
    visual_pos = torch.stack([time_ids, height_ids, width_ids], dim=0)

    # ── <|vision_end|>: sits just outside the grid corner ─────────
    # (3, 1)
    vis_end_pos = torch.tensor([[0], [H], [W]], dtype=torch.long, device=device)

    # ── Text tokens: all three dims increment together ─────────────
    # This degenerates to standard 1-D RoPE for text; crucial for
    # compatibility with a pretrained LLM whose 1-D RoPE we reuse.
    text_offset = max(H, W) + 1   # start well past the image grid
    text_range  = torch.arange(text_length, dtype=torch.long, device=device) + text_offset
    # (3, text_length)  — broadcast the 1-D positions to all 3 dims
    text_pos    = text_range.unsqueeze(0).expand(3, -1)

    # Concatenate all segments: (3, 1 + num_patches + 1 + text_length)
    position_ids = torch.cat([vis_start_pos, visual_pos, vis_end_pos, text_pos], dim=1)
    return position_ids


# ──────────────────────────────────────────────────────────────────
# Build cos/sin embeddings from 3D position IDs.
# head_dim is split: each third is modulated by one coordinate.
# ──────────────────────────────────────────────────────────────────
def build_mrope_cos_sin(
    position_ids: torch.Tensor,   # (3, seq_len)
    head_dim: int,
    base: float = 10000.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute cos and sin tensors for mRoPE application.

    Returns:
        cos : (seq_len, head_dim)
        sin : (seq_len, head_dim)
    """
    assert head_dim % 6 == 0, (
        f"head_dim {head_dim} must be divisible by 6 "
        f"(3 mRoPE components × 2 for rotate_half)"
    )
    device            = position_ids.device
    dim_per_component = head_dim // 3   # 16 for head_dim=48
    seq_len           = position_ids.shape[1]

    # Shared inverse frequencies (same for time / height / width)
    # (dim_per_component // 2,)
    inv_freq = get_rope_inv_freq(dim_per_component, base, device)

    cos_parts, sin_parts = [], []
    for coord_idx in range(3):
        # Extract this coordinate's positions: (seq_len,)
        pos = position_ids[coord_idx].float()

        # Outer product → (seq_len, dim_per_component // 2)
        freqs = torch.outer(pos, inv_freq)

        # Duplicate for rotate_half: (seq_len, dim_per_component)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos_parts.append(emb.cos())
        sin_parts.append(emb.sin())

    # Concatenate all 3 components along the feature axis
    # (seq_len, head_dim)
    cos = torch.cat(cos_parts, dim=-1)
    sin = torch.cat(sin_parts, dim=-1)
    return cos, sin


# ──────────────────────────────────────────────────────────────────
# Apply mRoPE to query and key tensors inside an attention layer.
# rotate_half implements the -x₂, x₁ "rotation" of each (x₁,x₂) pair.
# ──────────────────────────────────────────────────────────────────
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Split the last dimension in half, negate the second half, then swap.
    This implements the 2-D rotation matrix in RoPE.

    (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
    """
    half = x.shape[-1] // 2
    # (batch_num, num_heads, seq_len, half)
    x1 = x[..., :half]
    # (batch_num, num_heads, seq_len, half)
    x2 = x[..., half:]
    # (batch_num, num_heads, seq_len, head_dim)
    return torch.cat([-x2, x1], dim=-1)


def apply_mrope(
    q: torch.Tensor,    # (batch_num, num_heads, seq_len, head_dim)
    k: torch.Tensor,    # (batch_num, num_heads, seq_len, head_dim)
    cos: torch.Tensor,  # (seq_len, head_dim)
    sin: torch.Tensor,  # (seq_len, head_dim)
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Apply mRoPE rotation to Q and K before attention computation.
    Broadcast cos/sin over batch and head dimensions.
    """
    # (seq_len, head_dim) → (1, 1, seq_len, head_dim)  for broadcasting
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    # Modulate each (q, k) vector by its position-dependent rotation
    # (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
    q_rot = q * cos + rotate_half(q) * sin
    # (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
    k_rot = k * cos + rotate_half(k) * sin
    return q_rot, k_rot


# ── Demo: visualise position_ids and cos patterns ────────────────
pos_ids = compute_mrope_position_ids(cfg.patch_grid_hw, text_length=5)
cos_demo, sin_demo = build_mrope_cos_sin(pos_ids, cfg.head_dim)
seq_len  = pos_ids.shape[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Position IDs heatmap
im0 = axes[0].imshow(pos_ids.numpy(), aspect="auto", cmap="viridis")
axes[0].set_yticks([0, 1, 2]); axes[0].set_yticklabels(["time", "height", "width"])
axes[0].set_xlabel("Sequence position"); axes[0].set_title("mRoPE position_ids  (3 × seq_len)")
plt.colorbar(im0, ax=axes[0])

# Add region annotations
for label, x0, x1 in [
    ("<vis_start>", 0, 1), ("patches\n(4×4 grid)", 1, 17),
    ("<vis_end>", 17, 18), ("text", 18, seq_len),
]:
    mid = (x0 + x1) / 2
    axes[0].text(mid, 2.65, label, ha="center", va="bottom", fontsize=7,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

# cos embedding heatmap (first 12 dims)
im1 = axes[1].imshow(cos_demo[:, :12].numpy().T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[1].set_xlabel("Sequence position"); axes[1].set_ylabel("Feature dim")
axes[1].set_title("cos(mRoPE)  — first 12 of 48 dims")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()
print(f"position_ids shape : {tuple(pos_ids.shape)}")
print(f"cos / sin shape    : {tuple(cos_demo.shape)}")


## 5.4.1 Causal Language Model with mRoPE

The LLM uses mRoPE inside every attention layer. The key change vs. standard RoPE:

1. Compute `position_ids` of shape `(3, seq_len)` **once** before the forward pass.
2. Convert to `(cos, sin)` of shape `(seq_len, head_dim)`.
3. Inside each attention layer, apply `apply_mrope(q, k, cos, sin)` **before** computing the attention scores.

Everything else (FFN, residuals, layer norm) is identical to a standard transformer.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Multi-head attention with mRoPE.
# cos / sin are pre-computed outside and passed in for efficiency
# (avoids recomputing per layer).
# ──────────────────────────────────────────────────────────────────
class MRoPEAttention(nn.Module):
    def __init__(self, cfg: VLMConfig):
        super().__init__()
        self.num_heads = cfg.llm_num_heads
        self.head_dim  = cfg.head_dim
        self.scale     = self.head_dim ** -0.5

        self.q_proj   = nn.Linear(cfg.llm_dim, cfg.llm_dim, bias=False)
        self.k_proj   = nn.Linear(cfg.llm_dim, cfg.llm_dim, bias=False)
        self.v_proj   = nn.Linear(cfg.llm_dim, cfg.llm_dim, bias=False)
        self.out_proj = nn.Linear(cfg.llm_dim, cfg.llm_dim, bias=False)

    def forward(
        self,
        x: torch.Tensor,                        # (batch_num, seq_len, llm_dim)
        cos: torch.Tensor,                       # (seq_len, head_dim)
        sin: torch.Tensor,                       # (seq_len, head_dim)
        attn_mask: Optional[torch.Tensor] = None,# (batch_num, 1, seq_len, seq_len)
    ) -> torch.Tensor:
        batch_num, seq_len, _ = x.shape

        # Project into Q, K, V and reshape for multi-head computation
        # (batch_num, seq_len, llm_dim) → (batch_num, num_heads, seq_len, head_dim)
        q = self.q_proj(x).view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Inject position information via mRoPE rotation
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        q, k = apply_mrope(q, k, cos, sin)

        # Scaled dot-product attention with optional additive mask
        # (batch_num, num_heads, seq_len, seq_len)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        if attn_mask is not None:
            scores = scores + attn_mask   # attn_mask contains 0 or -inf
        attn_weights = F.softmax(scores, dim=-1)

        # Aggregate values; reshape back to sequence format
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, llm_dim)
        out = (attn_weights @ v).transpose(1, 2).reshape(batch_num, seq_len, -1)
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, llm_dim)
        return self.out_proj(out)


# ──────────────────────────────────────────────────────────────────
# Single transformer block for the LLM: pre-norm, attention + FFN.
# ──────────────────────────────────────────────────────────────────
class LLMBlock(nn.Module):
    def __init__(self, cfg: VLMConfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.llm_dim)
        self.attn  = MRoPEAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.llm_dim)
        # Standard GELU-FFN for the LLM backbone
        self.ffn   = nn.Sequential(
            nn.Linear(cfg.llm_dim, cfg.llm_ffn_dim),
            nn.GELU(),
            nn.Linear(cfg.llm_ffn_dim, cfg.llm_dim),
        )

    def forward(
        self,
        x: torch.Tensor,                         # (batch_num, seq_len, llm_dim)
        cos: torch.Tensor,                        # (seq_len, head_dim)
        sin: torch.Tensor,                        # (seq_len, head_dim)
        attn_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # Pre-norm attention residual block
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, llm_dim)
        x = x + self.attn(self.norm1(x), cos, sin, attn_mask)
        # Pre-norm FFN residual block
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, llm_dim)
        x = x + self.ffn(self.norm2(x))
        return x


# ──────────────────────────────────────────────────────────────────
# Tiny causal language model.
# Swap this class with Qwen2.5 / LLaMA from HuggingFace for production:
#
#   from transformers import AutoModelForCausalLM
#   llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
#
# The VisionLanguageModel forward pass below is interface-compatible.
# ──────────────────────────────────────────────────────────────────
class TinyLLM(nn.Module):
    """
    Minimal causal transformer with mRoPE.
    - No KV-cache (for clarity; trivial to add for inference)
    - Pre-norm architecture (matches Qwen2.5 / LLaMA design)
    """

    def __init__(self, cfg: VLMConfig):
        super().__init__()
        self.cfg        = cfg
        self.embed      = nn.Embedding(cfg.vocab_size, cfg.llm_dim, padding_idx=cfg.pad_token_id)
        self.blocks     = nn.ModuleList([LLMBlock(cfg) for _ in range(cfg.llm_num_layers)])
        self.norm       = nn.LayerNorm(cfg.llm_dim)
        self.lm_head    = nn.Linear(cfg.llm_dim, cfg.vocab_size, bias=False)

        # Weight-tying: lm_head shares weights with embed (reduces params, standard practice)
        self.lm_head.weight = self.embed.weight

    def get_embeddings(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Look up token embeddings (used by VLM to get the base embeddings
        before overwriting patch placeholder positions).

        Args:
            token_ids: (batch_num, seq_len)
        Returns:
            (batch_num, seq_len, llm_dim)
        """
        # (batch_num, seq_len) → (batch_num, seq_len, llm_dim)
        return self.embed(token_ids)

    def forward_from_embeddings(
        self,
        embeds: torch.Tensor,                    # (batch_num, seq_len, llm_dim)
        cos: torch.Tensor,                       # (seq_len, head_dim)
        sin: torch.Tensor,                       # (seq_len, head_dim)
        attn_mask: Optional[torch.Tensor] = None,# (batch_num, 1, seq_len, seq_len)
    ) -> torch.Tensor:
        """
        Forward pass starting from pre-computed embeddings (used by VLM
        after visual tokens have been injected at patch positions).

        Returns:
            logits: (batch_num, seq_len, vocab_size)
        """
        x = embeds
        for block in self.blocks:
            # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, llm_dim)
            x = block(x, cos, sin, attn_mask)
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, llm_dim)
        x = self.norm(x)
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, vocab_size)
        return self.lm_head(x)


llm = TinyLLM(cfg).to(DEVICE)
print(f"TinyLLM params: {sum(p.numel() for p in llm.parameters()):,}")


# 5.5 Forward Pass Walkthrough

## Attention Masking

The attention mask encodes two rules:

1. **Visual tokens → bidirectional**: every patch can attend to every other patch
   (the ViT already processed them as bidirectional; preserving this in the LLM
   allows cross-patch reasoning without causal restriction).

2. **Text tokens → causal + full visual**: each text token attends to all visual
   tokens and to all preceding text tokens (standard autoregressive decoding).

```
Sequence: [V_s  v0 v1 v2  V_e  t0 t1 t2]   (V_s=vis_start, V_e=vis_end)
          indices: 0  1  2  3   4   5  6  7

Attention matrix (1=attend, 0=mask):

           V_s v0 v1 v2 V_e t0 t1 t2
    V_s  [  1   1  1  1   1  0  0  0 ]   visual: attend within visual block
    v0   [  1   1  1  1   1  0  0  0 ]
    v1   [  1   1  1  1   1  0  0  0 ]
    v2   [  1   1  1  1   1  0  0  0 ]
    V_e  [  1   1  1  1   1  0  0  0 ]
    t0   [  1   1  1  1   1  1  0  0 ]   text: attend to all visual + causal text
    t1   [  1   1  1  1   1  1  1  0 ]
    t2   [  1   1  1  1   1  1  1  1 ]
```

## Loss Masking

Loss is computed **only on caption (assistant response) tokens**.
Image tokens and the user prompt are masked with `-100` in the label tensor
(PyTorch's `CrossEntropyLoss` ignores index `-100` by convention).


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Build the additive attention mask (0 = attend, -inf = block).
# Additive masks are added to the raw logits before softmax.
# ──────────────────────────────────────────────────────────────────
def build_attention_mask(
    seq_len: int,
    vis_start: int,
    vis_end: int,
    device: torch.device,
) -> torch.Tensor:
    """
    Construct the (1, 1, seq_len, seq_len) additive attention mask.

    Visual block : bidirectional (attend to all visual positions)
    Text block   : causal (attend to visual + previous text)

    Args:
        vis_start : index of <|vision_start|>
        vis_end   : index of <|vision_end|>   (inclusive)
    Returns:
        mask : (1, 1, seq_len, seq_len)
               0.0 for attended positions, -inf for masked
    """
    # Start with a fully causal lower-triangular mask for text
    # (seq_len, seq_len) — True means "attend"
    causal = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))

    # Visual tokens attend bidirectionally within the visual block
    # (vis_end+1 - vis_start) × (vis_end+1 - vis_start) all-ones patch
    causal[vis_start : vis_end + 1, vis_start : vis_end + 1] = True

    # Convert boolean mask to additive float mask
    # (seq_len, seq_len) → (1, 1, seq_len, seq_len)
    mask = torch.zeros(seq_len, seq_len, dtype=torch.float32, device=device)
    mask[~causal] = float("-inf")
    return mask.unsqueeze(0).unsqueeze(0)


# ──────────────────────────────────────────────────────────────────
# Visual injection: overwrite LLM embedding table lookups at patch
# placeholder positions with the actual projected visual features.
# ──────────────────────────────────────────────────────────────────
def inject_visual_tokens(
    token_embeds: torch.Tensor,     # (batch_num, seq_len, llm_dim)
    visual_embeds: torch.Tensor,    # (batch_num, num_patches, llm_dim)
    vis_start: int,
    vis_end: int,
) -> torch.Tensor:
    """
    Replace placeholder image_pad embeddings with projected visual features.

    The patch positions are [vis_start+1 .. vis_end-1] (exclusive of markers).
    We clone to avoid in-place modification of the autograd graph.

    Returns:
        mixed_embeds: (batch_num, seq_len, llm_dim)
    """
    mixed = token_embeds.clone()
    # Overwrite placeholder positions with projected patch features
    # (batch_num, num_patches, llm_dim)
    mixed[:, vis_start + 1 : vis_end, :] = visual_embeds
    return mixed


# ──────────────────────────────────────────────────────────────────
# Full VLM forward pass: combines ViT, projector, and LLM.
# ──────────────────────────────────────────────────────────────────
class VisionLanguageModel(nn.Module):
    """
    Qwen2.5-VL-style VLM.

    Stage 1 training:
        - Freeze ViT and LLM
        - Only projector.parameters() optimised

    Stage 2+ training:
        - Unfreeze LLM (and optionally ViT) — use LoRA on LLM for efficiency
    """

    def __init__(self, vit: TinyViT, projector: MLPProjector, llm: TinyLLM, cfg: VLMConfig):
        super().__init__()
        self.vit       = vit
        self.projector = projector
        self.llm       = llm
        self.cfg       = cfg

    def forward(
        self,
        pixel_values: torch.Tensor,       # (batch_num, 3, H, W)
        input_ids: torch.Tensor,          # (batch_num, seq_len)
        vis_ranges: torch.Tensor,         # (batch_num, 2)  — [vis_start, vis_end] per sample
        labels: Optional[torch.Tensor] = None,  # (batch_num, seq_len) — -100 to mask
    ) -> Dict[str, torch.Tensor]:
        """
        Returns dict with keys:
            logits : (batch_num, seq_len, vocab_size)
            loss   : scalar (only if labels provided)
        """
        batch_num = pixel_values.shape[0]

        # ── Step 1: encode images with frozen ViT ─────────────────
        # (batch_num, 3, H, W) → (batch_num, num_patches, visual_dim)
        visual_features = self.vit(pixel_values)

        # ── Step 2: project to LLM dimension ──────────────────────
        # (batch_num, num_patches, visual_dim) → (batch_num, num_patches, llm_dim)
        visual_embeds = self.projector(visual_features)

        # ── Step 3: get text token embeddings from LLM ────────────
        # (batch_num, seq_len) → (batch_num, seq_len, llm_dim)
        token_embeds = self.llm.get_embeddings(input_ids)

        # ── Step 4: overwrite placeholder positions with visual tokens
        # Assume uniform vis_ranges within a batch (standard in training)
        vis_start = int(vis_ranges[0, 0])
        vis_end   = int(vis_ranges[0, 1])
        # (batch_num, seq_len, llm_dim)
        mixed_embeds = inject_visual_tokens(token_embeds, visual_embeds, vis_start, vis_end)

        # ── Step 5: compute mRoPE position IDs and embeddings ─────
        seq_len  = input_ids.shape[1]
        text_len = seq_len - (vis_end + 1)  # tokens after <|vision_end|>
        # (3, seq_len)
        pos_ids  = compute_mrope_position_ids(self.cfg.patch_grid_hw, text_len, pixel_values.device)
        # (seq_len, head_dim), (seq_len, head_dim)
        cos, sin = build_mrope_cos_sin(pos_ids, self.cfg.head_dim, self.cfg.rope_base)

        # ── Step 6: build bidirectional-visual + causal-text mask ─
        # (1, 1, seq_len, seq_len)
        attn_mask = build_attention_mask(seq_len, vis_start, vis_end, pixel_values.device)

        # ── Step 7: LLM forward pass from mixed embeddings ─────────
        # (batch_num, seq_len, llm_dim) → (batch_num, seq_len, vocab_size)
        logits = self.llm.forward_from_embeddings(mixed_embeds, cos, sin, attn_mask)

        output = {"logits": logits}

        # ── Step 8: compute masked cross-entropy loss ──────────────
        if labels is not None:
            # Shift logits and labels for next-token prediction
            # (batch_num, seq_len-1, vocab_size)
            shift_logits = logits[:, :-1, :].contiguous()
            # (batch_num, seq_len-1)
            shift_labels = labels[:, 1:].contiguous()

            # Flatten and compute loss; -100 positions are automatically ignored
            # scalar
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
            output["loss"] = loss

        return output


vlm = VisionLanguageModel(vit, projector, llm, cfg).to(DEVICE)
print(f"VLM total params : {sum(p.numel() for p in vlm.parameters()):,}")


In [ ]:
# ──────────────────────────────────────────────────────────────────
# End-to-end forward pass trace on a single example.
# Validates that shapes, masking, and loss computation work correctly.
# ──────────────────────────────────────────────────────────────────
from functools import partial

collate_fn = partial(collate_vlm_batch, tokenizer=tokenizer, cfg=cfg)
loader     = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
batch      = next(iter(loader))

# Move batch to device
batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

vlm.eval()
with torch.no_grad():
    out = vlm(
        pixel_values=batch["pixel_values"],
        input_ids=batch["input_ids"],
        vis_ranges=batch["vis_ranges"],
        labels=batch["labels"],
    )

print("── Forward pass shapes ────────────────────────────────────────")
print(f"  pixel_values : {tuple(batch['pixel_values'].shape)}")
print(f"  input_ids    : {tuple(batch['input_ids'].shape)}")
print(f"  logits       : {tuple(out['logits'].shape)}")
print(f"  loss         : {out['loss'].item():.4f}")
print(f"  baseline NLL : {math.log(cfg.vocab_size):.4f}  (random guessing)")

# Visualise the attention mask pattern
seq_len   = batch["input_ids"].shape[1]
vis_start = int(batch["vis_ranges"][0, 0])
vis_end   = int(batch["vis_ranges"][0, 1])
mask      = build_attention_mask(seq_len, vis_start, vis_end, torch.device("cpu"))

fig, ax = plt.subplots(figsize=(7, 6))
attend = (mask[0, 0] == 0.0).float().numpy()
ax.imshow(attend, cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
ax.set_title("Attention mask  (blue = attend, white = blocked)")
# Annotate regions
for (label, x0, x1) in [("<vis>", 0, vis_end+1), ("text", vis_end+1, seq_len)]:
    ax.axvline(x0 - 0.5, color="red", linewidth=1.2, linestyle="--")
    ax.text((x0 + x1) / 2, -1.2, label, ha="center", fontsize=8, color="red")
plt.tight_layout()
plt.show()


# 5.6 Stage 1 Training Loop — Projector Only

## Rationale

In Stage 1 we **freeze everything except the MLP projector**.

- The ViT has already learned good visual representations (from SigLIP pretraining).
- The LLM has already learned good language representations.
- There is *no point* in disturbing these expensive, well-trained representations just to learn a 2-layer bridge.
- Focusing on the projector also means very few parameters to optimise (~200 K in our demo), so convergence is fast.

**Training objective**: next-token prediction on captions, conditioned on visual tokens.

$$\mathcal{L} = -\sum_{t} \log p_\theta\left(w_t \mid w_{<t},\, v_{1:N}\right)$$

where $v_{1:N}$ are the projected visual tokens.

## Gradient flow

```
image → ViT (frozen) → projector (grad ✓) → LLM (frozen) → loss
```

Only `projector.parameters()` receive gradients.
The LLM and ViT are in `torch.no_grad()` mode via `requires_grad=False`.


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Freeze ViT and LLM; only projector remains trainable (Stage 1).
# ──────────────────────────────────────────────────────────────────
def set_stage1_trainability(model: VisionLanguageModel) -> None:
    """Freeze all parameters except the MLP projector."""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.projector.parameters():
        param.requires_grad = True


set_stage1_trainability(vlm)

trainable   = sum(p.numel() for p in vlm.parameters() if p.requires_grad)
total       = sum(p.numel() for p in vlm.parameters())
print(f"Trainable params : {trainable:,}  /  {total:,}  ({100*trainable/total:.1f}%)")
print(f"All projector params trainable: "
      f"{all(p.requires_grad for p in vlm.projector.parameters())}")
print(f"All LLM params frozen         : "
      f"{all(not p.requires_grad for p in vlm.llm.parameters())}")


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Stage 1 training loop.
# Uses AdamW with a linear warmup + cosine decay schedule.
# ──────────────────────────────────────────────────────────────────
def get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
) -> torch.optim.lr_scheduler.LambdaLR:
    """Linear warmup followed by cosine decay."""

    def lr_lambda(step: int) -> float:
        if step < num_warmup_steps:
            return float(step) / max(1, num_warmup_steps)
        progress = float(step - num_warmup_steps) / max(1, num_training_steps - num_warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_stage1(
    vlm: VisionLanguageModel,
    dataset: SyntheticColorDataset,
    tokenizer,
    cfg: VLMConfig,
    num_epochs: int = 15,
    batch_size: int = 16,
    lr: float = 1e-3,
    warmup_ratio: float = 0.1,
) -> Dict[str, List[float]]:
    """
    Stage 1 training: align visual features to the LLM's embedding space.

    Returns:
        history : {'train_loss': [...], 'lr': [...]}
    """
    collate_fn = partial(collate_vlm_batch, tokenizer=tokenizer, cfg=cfg)
    loader     = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            collate_fn=collate_fn, drop_last=True)

    # Only pass projector parameters to the optimiser
    optimizer = torch.optim.AdamW(
        vlm.projector.parameters(), lr=lr, weight_decay=0.01
    )

    total_steps  = num_epochs * len(loader)
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    history: Dict[str, List[float]] = {"train_loss": [], "lr": []}
    vlm.train()

    for epoch in range(num_epochs):
        epoch_loss, n_batches = 0.0, 0

        for batch in loader:
            # Move tensors to device
            batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}

            # Forward pass (ViT + LLM are frozen → no-grad internally for those params)
            out = vlm(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                vis_ranges=batch["vis_ranges"],
                labels=batch["labels"],
            )
            loss = out["loss"]

            # Gradient update on projector parameters only
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vlm.projector.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            n_batches  += 1

        avg_loss = epoch_loss / n_batches
        cur_lr   = scheduler.get_last_lr()[0]
        history["train_loss"].append(avg_loss)
        history["lr"].append(cur_lr)

        if (epoch + 1) % 3 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs} | loss={avg_loss:.4f} | lr={cur_lr:.2e}")

    return history


# Run Stage 1 training
history = train_stage1(vlm, dataset, tokenizer, cfg)


In [ ]:
# ── Plot training curves ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], color="steelblue", linewidth=2)
ax1.axhline(math.log(cfg.vocab_size), color="red", linestyle="--", label="Random-guess NLL")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Stage 1 Training Loss (projector only)")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history["lr"], color="darkorange", linewidth=2)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Learning rate")
ax2.set_title("Cosine LR Schedule with Warmup")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Final loss : {history['train_loss'][-1]:.4f}")
print(f"Baseline   : {math.log(cfg.vocab_size):.4f}  (random guessing)")


# 5.7 Verification: Greedy Caption Generation

## Intuition

After Stage 1, the projector has learned to map ViT patch features into the LLM's
embedding space. We verify this by doing *greedy token generation*:

1. Encode the image → project visual tokens.
2. Start with the sequence `[<vis_start>, v₁…vₙ, <vis_end>]`.
3. Autoregressively sample the next token (argmax for greedy decoding).
4. Append and repeat until EOS or `max_new_tokens`.

If the model learned the alignment, it should generate color words that match
the input image, even though the LLM weights were **never updated**.


In [ ]:
@torch.no_grad()
def generate_caption(
    vlm: VisionLanguageModel,
    image: torch.Tensor,            # (3, H, W) single image
    tokenizer,
    cfg: VLMConfig,
    max_new_tokens: int = 10,
) -> str:
    """
    Greedy autoregressive caption generation for a single image.

    Steps:
        1. Build initial sequence (visual tokens only, no text prefix).
        2. At each step: forward pass → take argmax of last logit → append.
        3. Stop at EOS token or max_new_tokens.
    """
    vlm.eval()
    device = next(vlm.parameters()).device

    # Add batch dimension for model compatibility
    # (3, H, W) → (1, 3, H, W)
    pixel_values = image.unsqueeze(0).to(device)

    # Build initial token sequence: [<vis_start> pad×N <vis_end>]
    vis_ids = (
        [cfg.vision_start_id]
        + [cfg.image_pad_id] * cfg.num_patches
        + [cfg.vision_end_id]
    )
    # (1, vis_seq_len)
    current_ids = torch.tensor([vis_ids], dtype=torch.long, device=device)
    vis_start, vis_end = 0, len(vis_ids) - 1

    generated_ids: List[int] = []

    for _ in range(max_new_tokens):
        vis_ranges = torch.tensor([[vis_start, vis_end]], device=device)

        # Full forward pass on current sequence
        out    = vlm(pixel_values=pixel_values, input_ids=current_ids, vis_ranges=vis_ranges)
        # Take argmax of the last-position logit
        # (1, seq_len, vocab_size) → (1,)
        next_id = out["logits"][:, -1, :].argmax(dim=-1)

        # Stop if end-of-sequence token
        if next_id.item() == tokenizer.sep_token_id:
            break

        generated_ids.append(next_id.item())
        # Append predicted token to running sequence
        # (1, seq_len) → (1, seq_len + 1)
        current_ids = torch.cat([current_ids, next_id.unsqueeze(1)], dim=1)

    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# ── Evaluate on held-out examples ────────────────────────────────
test_colors = ["red", "blue", "green", "yellow", "purple",
               "orange", "pink", "cyan", "white", "gray"]

print(f"{'Color':<10}  {'Generated caption':<50}  Match?")
print("─" * 70)

correct = 0
for color in test_colors:
    img     = dataset._make_image(color)
    caption = generate_caption(vlm, img, tokenizer, cfg, max_new_tokens=8)
    match   = color in caption
    correct += int(match)
    mark    = "✓" if match else "✗"
    print(f"{color:<10}  {caption:<50}  {mark}")

print(f"\nAccuracy: {correct}/{len(test_colors)}")


In [ ]:
# ── Visual verification grid ─────────────────────────────────────
# Show images alongside their model-generated captions.
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, color in zip(axes.flat, test_colors):
    img     = dataset._make_image(color)
    caption = generate_caption(vlm, img, tokenizer, cfg, max_new_tokens=8)

    ax.imshow(img.permute(1, 2, 0).numpy())
    wrapped = caption[:35] + "…" if len(caption) > 35 else caption
    match   = color in caption
    border  = "green" if match else "red"

    for spine in ax.spines.values():
        spine.set_edgecolor(border); spine.set_linewidth(3)

    ax.set_title(f"GT: {color}\n\"{wrapped}\"", fontsize=8, pad=3)
    ax.axis("off")

plt.suptitle("Stage 1 Verification: Generated Captions (green border = correct color)", y=1.01)
plt.tight_layout()
plt.show()


# Summary & Next Steps

## What we built

| Notebook section | Component | Key insight |
|---|---|---|
| 5.1 | TinyViT + MLPProjector | SwiGLU FFN; no CLS token; 2-layer projector only trainable in S1 |
| 5.2 | Synthetic data pipeline | Dynamic resolution; token counts scale as H×W/p² |
| 5.3 | Visual token insertion | `<\|vision_start\|>` + placeholder IDs + `<\|vision_end\|>` |
| 5.4 | mRoPE | (3, seq_len) position IDs; head_dim split into time/height/width |
| 5.5 | Attention + loss masks | Bidirectional within visual; causal for text; -100 label masking |
| 5.6 | Stage 1 training | Freeze ViT + LLM; optimise projector only; ~minutes on CPU |
| 5.7 | Verification | Greedy generation checks alignment quality |

## Production differences (Qwen2.5-VL scale)

| This notebook | Qwen2.5-VL |
|---|---|
| 64×64 images, 16 patches | Up to 1344×896, 6144 patches |
| TinyViT (256-d, 3 layers) | SigLIP ViT-G (1152-d, 48 layers) |
| TinyLLM (192-d, 4 layers) | Qwen2.5-7B / 32B / 72B |
| Synthetic 300 samples | 600K+ image-caption pairs |
| ~200K projector params | ~10M projector params |

## Chapter 6 preview

Chapter 6 covers **Stage 2–3 training**: unfreezing the LLM, multi-turn conversation
templates, LoRA adaptation, DPO for hallucination reduction, and RL-based
visual reasoning (GRPO with verifier rewards — the multimodal DeepSeek-R1).
